In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

import numpy as np 
from src.dataset import Dataset
from src.logistic_regression import LogisticRegression
import matplotlib.pyplot as pyplot
import pandas as pd
from collections import Counter
import re


In [2]:
def build_vocab(texts, max_features = 5000, min_freq = 2):
    counter = Counter()

    for text in texts:
        counter.update(text.split())

    vocab_items = [
        word for word, freq in counter.items()
        if freq >= min_freq
    ]

    vocab_items = sorted(
        vocab_items,
        key= lambda word: counter[word],
        reverse= True
    ) [:max_features]

    vocab = {word: idx for idx, word in enumerate(vocab_items)}
    return vocab

In [3]:
def text_to_bow(texts, vocab):
    X = np.zeros((len(texts), len(vocab)), dtype=np.float32)
    
    for i, text in enumerate(texts):
        for word in text.split():
            if word in vocab:
                X[i, vocab[word]] += 1.0
                
    return X

In [4]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [5]:
def fit_label_encoder(labels):
    classes = sorted(labels.unique())
    label_to_idx = {label: i for i, label in enumerate(classes)}
    idx_to_label = {i: label for label, i in label_to_idx.items()}
    return label_to_idx, idx_to_label

def encode_labels(labels, label_to_idx):
    return np.array([label_to_idx[label] for label in labels], dtype=np.int32)

In [6]:
# def one_hot_encode(y, num_classes):
#     y_onehot = np.zeros((len(y), num_classes), dtype=np.float32)
#     y_onehot[np.arange(len(y)), y] = 1.0
#     return y_onehot

In [7]:
def train_val_split(X, y, val_size=0.2, seed=42):
    np.random.seed(seed)
    indices = np.arange(len(X))
    np.random.shuffle(indices)
    
    split_idx = int(len(X) * (1 - val_size))
    train_idx = indices[:split_idx]
    val_idx = indices[split_idx:]
    
    return X[train_idx], X[val_idx], y[train_idx], y[val_idx]

In [8]:
PATH = "../datasets/dataset-exemplos.csv"

df = pd.read_csv(PATH, sep=";")
df["clean_text"] = df["Text"].apply(clean_text)

vocab = build_vocab(df["clean_text"], max_features=5000, min_freq=2)
X_all = text_to_bow(df["clean_text"], vocab)

label_to_idx, idx_to_label = fit_label_encoder(df["Label"])
y_idx = encode_labels(df["Label"], label_to_idx)
# num_classes = len(label_to_idx)
# y_all = one_hot_encode(y_idx, num_classes)

In [9]:
X_train, X_val, y_train, y_val = train_val_split(X_all, y_idx, val_size=0.2, seed=42)

dataset = Dataset(X_train, Y=y_train)
logmodel = LogisticRegression(dataset) 

(100, 1542)


In [10]:
print ("Initial cost: ", logmodel.costFunction())
logmodel.gradientDescent(0.002, 50000)
print ("Final cost:", logmodel.costFunction())

Initial cost:  1.6094379124341003
1.5898984793285837
0.49978171713684105
0.2720756553008791
0.17966965690962464
0.13192400894054593
0.10338387048206928
0.08461624879222111
0.07142374188435609
0.061684347391636
0.05422003548289577
0.04832840211130266
0.04356640315224011
0.03964160290879386
0.03635361778773248
0.03356076912980278
0.031160182259435433
0.029075429255680504
0.027248590032895814
0.02563499896442356
0.024199677419019138
0.022914855140505393
0.02175821275921314
0.02071161273412271
0.019760167839958473
0.01889154720037039
0.018095452269777143
0.01736321624072133
0.01668749432827461
0.01606202181520147
0.015481423211955996
0.014941060391022516
0.0144369107359521
0.013965468620046204
0.013523665175653727
0.013108802519456689
0.012718499489573352
0.012350646615051421
0.012003368539103571
0.011674992497931342
0.011364021748419818
0.011069113062908835
0.01078905758408636
0.010522764469882968
0.010269246866027024
0.010027609829334634
0.009797039892893511
0.009576796018871368
0.009366

In [11]:

# print ("Prob. example:", logmodel.probability(X_val))
print ("Pred. example:", logmodel.predict(X_val, y_val, len(X_val)))


Pred. example: 28.000000000000004
